In [1]:
!pip install simpy -q

import simpy
import random
import statistics

# Simulation settings
RANDOM_SEED = 42
NUM_PACKETS = 50
PACKET_INTERARRIVAL = 2.0
NUM_ROUTERS = 3
PROCESSING_TIME_MEAN = 1.0
SIM_TIME = 300

random.seed(RANDOM_SEED)

class Router:
    def __init__(self, env, name, processing_mean):
        self.env = env
        self.name = name
        self.processing_mean = processing_mean
        self.resource = simpy.Resource(env, capacity=1)
        self.packets_processed = 0

    # Simulate processing a packet
    def process_packet(self, packet):
        delay = random.expovariate(1.0 / self.processing_mean)
        yield self.env.timeout(delay)
        self.packets_processed += 1

# Move a packet through each router
def packet_journey(env, packet_id, routers, results):
    start_time = env.now

    for router in routers:
        with router.resource.request() as req:
            yield req
            yield env.process(router.process_packet(packet_id))

    end_time = env.now

    results.append({
        "packet_id": packet_id,
        "start_time": start_time,
        "end_time": end_time,
        "latency": end_time - start_time
    })

# Generate packets at random times
def packet_generator(env, routers, results):
    for packet_id in range(1, NUM_PACKETS + 1):
        yield env.timeout(random.expovariate(1.0 / PACKET_INTERARRIVAL))
        env.process(packet_journey(env, packet_id, routers, results))

# Create the simulation
env = simpy.Environment()

routers = [
    Router(env, f"Router-{chr(65+i)}", PROCESSING_TIME_MEAN)
    for i in range(NUM_ROUTERS)
]

results = []

env.process(packet_generator(env, routers, results))
env.run(until=SIM_TIME)

# Calculate latency statistics
latencies = [r["latency"] for r in results]

print(f"Packets delivered: {len(results)}")

for router in routers:
    print(f"{router.name}: processed {router.packets_processed} packets")

print(f"Mean latency: {statistics.mean(latencies):.3f}s")
print(f"Min/Max latency: {min(latencies):.3f}s / {max(latencies):.3f}s")
print(f"Throughput: {len(results)/max(r['end_time'] for r in results):.3f} packets/sec")

Packets delivered: 50
Router-A: processed 50 packets
Router-B: processed 50 packets
Router-C: processed 50 packets
Mean latency: 6.502s
Min/Max latency: 1.442s / 16.787s
Throughput: 0.444 packets/sec
